In [18]:
##### Tests alternative models (LightGBM and XGBoost)

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from step_a_config import RunConfig
from step_b_spatial_CV import make_spatial_folds
from step_c_train import compute_sample_weights

In [19]:
# ── SET-UP DIRECTORIES (same layout as original RF notebook) ───────────────

cd = Path.cwd().parent.parent
DATA_DIR = Path(f"{cd}/Data/Clean/Training_data/")
FOLD_DIR = Path(f"{cd}/Data/Fold_assignments/")

In [20]:
# ── FIXED HYPERPARAMETERS (no search -- reasonable defaults for speed) ─────

LGBM_PARAMS = {
    "n_estimators":  500,
    "num_leaves":    31,
    "max_depth":     -1,
    "learning_rate": 0.05,
    "subsample":     0.8,
    "colsample_bytree": 0.8,
}

XGB_PARAMS = {
    "n_estimators":  500,
    "max_depth":     6,
    "learning_rate": 0.05,
    "subsample":     0.8,
    "colsample_bytree": 0.8,
}

In [21]:
# ── FEATURE COLUMNS (same RFE-selected sets used for the RF models) ────────

capital_cols = ['rtv_log_average_travel_time_port',
       'rtv_log_crop_intensity',
       'rtv_log_USD_production_per_million_HA',
       'rtv_log_tonnes_production_per_million_HA',
       'rtv_log_pop_density_people_per_100_km2',
       'rtv_log_cattle_density_per_100_km2',
       'rtv_log_sheep_density_per_100_km2',
       'rtv_log_livestock_density_LU_per_100_km2',
       'rtv_log_cereals_share_base_100_production_tonnes',
       'rtv_log_fruits_share_base_100_production_tonnes',
       'rtv_log_roots_tubers_share_base_100_production_tonnes',
       'rtv_log_vegetables_share_base_100_production_tonnes',
       'rtv_log_ruminants_share_base_100_production_tonnes',
       'rtv_log_share_base_100_large_field',
       'rtv_log_share_base_100_with_nightlights']

labor_cols = ['rtv_log_average_travel_time_port',
       'rtv_log_crop_intensity',
       'rtv_log_pop_density_people_per_100_km2',
       'rtv_log_cattle_density_per_100_km2',
       'rtv_log_livestock_density_LU_per_100_km2',
       'rtv_log_fruits_share_base_100_production_tonnes',
       'rtv_log_roots_tubers_share_base_100_production_tonnes',
       'rtv_log_rest_of_crops_share_base_100_production_tonnes',
       'rtv_log_sugar_crops_share_base_100_production_tonnes',
       'rtv_log_ruminants_share_base_100_production_tonnes',
       'rtv_log_pct_base_100_GDP_ag', 'rtv_log_share_base_100_large_field',
       'rtv_log_pct_base_100_cropland_irrigated']

In [22]:
# ── MODEL ────────────────────────────────────────────────────────

def get_model(config):
    """Initialize a LightGBM or XGBoost regressor with fixed hyperparameters."""
    if config.model_type == "lgbm":
        return LGBMRegressor(**LGBM_PARAMS, random_state=config.random_seed, n_jobs=-1, verbosity=-1)
    elif config.model_type == "xgb":
        return XGBRegressor(**XGB_PARAMS, random_state=config.random_seed, n_jobs=-1, verbosity=0)
    else:
        raise ValueError(f"Unknown model type: {config.model_type} (expected 'lgbm' or 'xgb')")


# ── SPATIAL-CV TRAINING LOOP (fit once per fold, no inner search) ──────────

def train_model(df, folds, config):
    """
    Fits config.model_type (lgbm or xgb) once per spatial fold -- no inner
    hyperparameter search -- using the same train/test country splits as
    the RF pipeline. Computes R2 on the held-out test countries per fold.
    """
    X = df[config.feature_cols]
    y = df[config.target]

    fold_r2 = []

    for fold in folds:
        X_train = X.loc[fold["train_idx"]]
        y_train = y.loc[fold["train_idx"]]
        X_test  = X.loc[fold["test_idx"]]
        y_test  = y.loc[fold["test_idx"]]

        train_countries = df.loc[fold["train_idx"], "country_ID"]
        sample_weight   = compute_sample_weights(train_countries, config.weighting)

        model = get_model(config)
        model.fit(X_train, y_train, sample_weight=sample_weight)

        preds = model.predict(X_test)
        r2 = r2_score(y_test, preds)
        fold_r2.append(r2)
        print(f"  {fold['fold']}: R2 = {r2:.4f}")

    mean_r2 = float(np.mean(fold_r2))
    var_r2  = float(np.var(fold_r2, ddof=1)) if len(fold_r2) > 1 else 0.0

    return {"fold_r2": fold_r2, "mean_r2": mean_r2, "var_r2": var_r2}


# ── DEFINE RUNS: one per (target x alt model type) ──────────────────────────

RUNS = [
    RunConfig(
        run_name         = 'capital_lgbm_spatial_CV',
        target           = 'rtv_log_capital_intensity_USD_per_million_tonne',
        dataset          = "capital_relative_final_thinned.csv",
        fold_assignments = FOLD_DIR / "capital_folds.csv",
        model_type       = 'lgbm',
        feature_cols     = capital_cols,
        version          = 'capital',
    ),
    RunConfig(
        run_name         = 'capital_xgb_spatial_CV',
        target           = 'rtv_log_capital_intensity_USD_per_million_tonne',
        dataset          = "capital_relative_final_thinned.csv",
        fold_assignments = FOLD_DIR / "capital_folds.csv",
        model_type       = 'xgb',
        feature_cols     = capital_cols,
        version          = 'capital',
    ),
    RunConfig(
        run_name         = 'labor_lgbm_spatial_CV',
        target           = 'rtv_log_labor_intensity_jobs_per_million_tonne',
        dataset          = "labor_relative_final_thinned.csv",
        fold_assignments = FOLD_DIR / "labor_folds.csv",
        model_type       = 'lgbm',
        feature_cols     = labor_cols,
        version          = 'labor',
    ),
    RunConfig(
        run_name         = 'labor_xgb_spatial_CV',
        target           = 'rtv_log_labor_intensity_jobs_per_million_tonne',
        dataset          = "labor_relative_final_thinned.csv",
        fold_assignments = FOLD_DIR / "labor_folds.csv",
        model_type       = 'xgb',
        feature_cols     = labor_cols,
        version          = 'labor',
    ),
]


# ── RUN ──────────────────────────────────────────────────────────────────

def run(config):
    print(f"\n{'═'*60}")
    print(f"  run: {config.run_name}")
    print(f"{'═'*60}")

    df = pd.read_csv(DATA_DIR / config.dataset)
    folds = make_spatial_folds(df, config)
    results = train_model(df, folds, config)

    print(f"  mean R2 = {results['mean_r2']:.4f}   var R2 = {results['var_r2']:.6f}")
    return results


if __name__ == "__main__":
    all_results = {}
    for config in RUNS:
        all_results[config.run_name] = run(config)

    print(f"\n{'═'*60}")
    print("  SUMMARY: R2 across spatial CV folds")
    print(f"{'═'*60}")
    for run_name, res in all_results.items():
        print(f"  {run_name:30s}  mean R2 = {res['mean_r2']:.4f}   var R2 = {res['var_r2']:.6f}")


════════════════════════════════════════════════════════════
  run: capital_lgbm_spatial_CV
════════════════════════════════════════════════════════════
  fold_1: 18 train countries (1,891 rows) | 1 test countries (852 rows)
  fold_2: 18 train countries (1,690 rows) | 1 test countries (1,053 rows)
  fold_3: 18 train countries (2,443 rows) | 1 test countries (300 rows)
  fold_4: 15 train countries (2,474 rows) | 4 test countries (269 rows)
  fold_5: 7 train countries (2,474 rows) | 12 test countries (269 rows)
  fold_1: R2 = 0.4995
  fold_2: R2 = 0.0687
  fold_3: R2 = 0.2517
  fold_4: R2 = 0.3808
  fold_5: R2 = -0.1557
  mean R2 = 0.2090   var R2 = 0.067119

════════════════════════════════════════════════════════════
  run: capital_xgb_spatial_CV
════════════════════════════════════════════════════════════
  fold_1: 18 train countries (1,891 rows) | 1 test countries (852 rows)
  fold_2: 18 train countries (1,690 rows) | 1 test countries (1,053 rows)
  fold_3: 18 train countries (2,443